# Objetivo A Deteccion de Fraude en Apps Moviles con LightGBM

Este notebook implementa el Objetivo A para clasificacion binaria de fraude con metricas personalizadas de LightGBM enfocadas en reducir alertas falsas positivas en transacciones de apps moviles.

Columna objetivo: is_fraud

Metrica principal del negocio: false_positive_ratio igual a FP dividido entre TP mas FP.

## Configuracion del proyecto

Esta seccion importa librerias, fija la semilla aleatoria, crea la carpeta outputs y define constantes globales.

In [2]:
from pathlib import Path
import json
import pickle
import re
import unicodedata
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Global reproducibility settings
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Project paths
PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_CHECKS_DIR = OUTPUT_DIR / "01_data_checks"
EDA_DIR = OUTPUT_DIR / "02_eda"
FEATURES_DIR = OUTPUT_DIR / "03_features"
BASELINE_DIR = OUTPUT_DIR / "04_baseline"
CUSTOM_METRICS_DIR = OUTPUT_DIR / "05_custom_metrics"
TUNING_DIR = OUTPUT_DIR / "06_tuning"
FINAL_MODEL_DIR = OUTPUT_DIR / "07_final_model"
PLOTS_DIR = OUTPUT_DIR / "08_plots"
DELIVERY_DIR = OUTPUT_DIR / "09_delivery"
OUTPUT_INDEX_PATH = OUTPUT_DIR / "output_index.csv"

OUTPUT_SUBDIRS = {
    "data_checks": DATA_CHECKS_DIR,
    "eda": EDA_DIR,
    "features": FEATURES_DIR,
    "baseline": BASELINE_DIR,
    "custom_metrics": CUSTOM_METRICS_DIR,
    "tuning": TUNING_DIR,
    "final_model": FINAL_MODEL_DIR,
    "plots": PLOTS_DIR,
    "delivery": DELIVERY_DIR,
}


# Crea las carpetas de salida
def ensure_output_dirs():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for output_dir in OUTPUT_SUBDIRS.values():
        output_dir.mkdir(parents=True, exist_ok=True)


ensure_output_dirs()

# Main constants
# Define constantes principales
DATASET_FILE_NAME = "01_bo_vip_seed22_n100000.csv"
DATASET_PATTERN = "*01_bo_vip_seed22_n100000*.csv"
TARGET_COLUMN = "is_fraud"
ASSUMED_YEAR = 2025
TARGET_RECALL = 0.90
MOBILE_APP_COLUMN = "is_mobile_app_txn"

# Plot settings
# Configura graficas
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
warnings.filterwarnings("ignore")

print(f"Project directory: {PROJECT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


Project directory: d:\PLUSTI PROYECTO
Output directory: d:\PLUSTI PROYECTO\outputs


## Descubrimiento de archivos

Esta seccion lista los archivos del proyecto, identifica el dataset principal y muestra vistas previas de archivos de metadatos cuando existen.

In [ ]:
# Lista los archivos del proyecto
def get_file_inventory(project_dir):
    rows = []
    for path in sorted(project_dir.iterdir()):
        if path.is_file():
            rows.append(
                {
                    "file_name": path.name,
                    "suffix": path.suffix.lower(),
                    "size_bytes": path.stat().st_size,
                }
            )
    return pd.DataFrame(rows)


# Lee texto y lo convierte a ascii
def read_text_as_ascii(path, max_chars=5000):
    raw_bytes = path.read_bytes()
    for encoding_name in ["utf-8", "cp1252", "latin1"]:
        try:
            text = raw_bytes.decode(encoding_name)
            break
        except UnicodeDecodeError:
            text = ""
    if not text:
        text = raw_bytes.decode("latin1", errors="ignore")
    clean_text = text.encode("ascii", errors="ignore").decode("ascii")
    return clean_text[:max_chars]


# Muestra una vista previa de archivos tabulares
def preview_tabular_file(path, max_rows=5):
    try:
        if path.suffix.lower() == ".csv":
            sample = path.read_text(errors="ignore")[:4096]
            delimiter = ";" if sample.count(";") >= sample.count(",") else ","
            return pd.read_csv(path, sep=delimiter, nrows=max_rows).to_string()
        if path.suffix.lower() in [".xlsx", ".xls"]:
            return pd.read_excel(path, nrows=max_rows).to_string()
    except Exception as error:
        return f"Could not preview tabular file: {error}"
    return ""


# Muestra una vista previa de archivos pdf
def preview_pdf_file(path, max_chars=3000):
    try:
        import pypdf

        reader = pypdf.PdfReader(str(path))
        text_parts = []
        for page in reader.pages[:3]:
            text_parts.append(page.extract_text() or "")
        text = "\n".join(text_parts)
        return text.encode("ascii", errors="ignore").decode("ascii")[:max_chars]
    except Exception as error:
        return f"PDF preview unavailable: {error}"


# Muestra una vista previa de metadata
def preview_metadata_file(path):
    suffix = path.suffix.lower()
    if suffix in [".csv", ".xlsx", ".xls"]:
        return preview_tabular_file(path)
    return ""


file_inventory = get_file_inventory(PROJECT_DIR)
display(file_inventory)

metadata_suffixes = { ".csv", ".xlsx", ".xls"}


# Detecta archivos de transacciones
def is_likely_transaction_dataset(file_name):
    lower_name = file_name.lower()
    return lower_name.endswith(".csv") and "seed" in lower_name and "n100000" in lower_name


metadata_files = [
    PROJECT_DIR / file_name
    for file_name in file_inventory["file_name"].tolist()
    if (PROJECT_DIR / file_name).suffix.lower() in metadata_suffixes
    and "01_bo_vip_seed22_n100000" not in file_name
    and not is_likely_transaction_dataset(file_name)
]

,file_name,suffix,size_bytes
0,Copia de 01_bo_vip_seed22_n100000.csv,.csv,56111783
1,Copia de 02_br_privado_seed33_n100000.csv,.csv,56588931
2,Copia de 03_gt_estatal_seed3_n100000.csv,.csv,55910287
3,Descripciones de variables ISO 8583 Para Alumn...,.txt,11334
4,Indicaciones Trabajo Practico 1 UDV.pdf,.pdf,258487
5,ObjetivoA.ipynb,.ipynb,368883
6,Regla.txt,.txt,185
7,requirements.txt,.txt,126


Metadata files found:


## Carga de datos

Esta seccion carga el archivo CSV principal, normaliza los nombres de columnas a snake case y valida la columna objetivo.

In [4]:
# Convierte texto a ascii
def normalize_to_ascii(value):
    text = str(value)
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", errors="ignore").decode("ascii")
    return text


# Convierte texto a snake case
def to_snake_case(value):
    text = normalize_to_ascii(value)
    text = text.strip()
    text = re.sub(r"[^0-9a-zA-Z]+", "_", text)
    text = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", text)
    text = re.sub(r"_+", "_", text)
    text = text.strip("_").lower()
    return text or "unnamed_column"


# Normaliza nombres de columnas
def normalize_column_names(columns):
    normalized_columns = []
    seen_counts = {}
    for column in columns:
        base_name = to_snake_case(column)
        count = seen_counts.get(base_name, 0)
        if count == 0:
            normalized_columns.append(base_name)
        else:
            normalized_columns.append(f"{base_name}_{count + 1}")
        seen_counts[base_name] = count + 1
    return normalized_columns


# Busca el dataset principal
def find_dataset_path(project_dir, dataset_file_name, dataset_pattern):
    exact_path = project_dir / dataset_file_name
    if exact_path.exists():
        return exact_path
    matches = sorted(project_dir.glob(dataset_pattern))
    if matches:
        warnings.warn(f"Exact dataset name not found. Using fallback file: {matches[0].name}")
        return matches[0]
    raise FileNotFoundError(f"Dataset file not found with pattern: {dataset_pattern}")


# Detecta el separador del csv
def detect_csv_separator(path):
    sample = path.read_text(errors="ignore")[:8192]
    counts = {separator: sample.count(separator) for separator in [";", ",", "\t", "|"]}
    best_separator = max(counts, key=counts.get)
    return best_separator if counts[best_separator] > 0 else ","


# Convierte el target a binario
def convert_target_to_int(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(int)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(int)
    normalized = series.astype(str).str.strip().str.lower()
    mapping = {
        "true": 1,
        "false": 0,
        "1": 1,
        "0": 0,
        "yes": 1,
        "no": 0,
        "y": 1,
        "n": 0,
        "fraud": 1,
        "legit": 0,
        "legitimate": 0,
    }
    converted = normalized.map(mapping)
    if converted.isna().any():
        unknown_values = sorted(normalized[converted.isna()].dropna().unique().tolist())[:10]
        raise ValueError(f"Target has unsupported values: {unknown_values}")
    return converted.astype(int)


# Carga y valida el dataset
dataset_path = find_dataset_path(PROJECT_DIR, DATASET_FILE_NAME, DATASET_PATTERN)
csv_separator = detect_csv_separator(dataset_path)

print(f"Dataset path: {dataset_path}")
print(f"Detected separator: {repr(csv_separator)}")

raw_data = pd.read_csv(dataset_path, sep=csv_separator, low_memory=False)
original_columns = list(raw_data.columns)
raw_data.columns = normalize_column_names(raw_data.columns)
column_name_mapping = dict(zip(original_columns, raw_data.columns))

print(f"Loaded shape: {raw_data.shape}")
display(raw_data.head())

if TARGET_COLUMN not in raw_data.columns:
    raise ValueError(f"Required target column not found: {TARGET_COLUMN}")

raw_data[TARGET_COLUMN] = convert_target_to_int(raw_data[TARGET_COLUMN])
target_values = sorted(raw_data[TARGET_COLUMN].dropna().unique().tolist())
if target_values != [0, 1]:
    raise ValueError(f"Target must be binary 0 and 1. Found values: {target_values}")

with open(DATA_CHECKS_DIR / "column_name_mapping.json", "w", encoding="utf-8") as file:
    json.dump(column_name_mapping, file, indent=2, ensure_ascii=True)

print("Target validation passed")
print(raw_data[TARGET_COLUMN].value_counts(dropna=False).to_string())

Dataset path: d:\PLUSTI PROYECTO\Copia de 01_bo_vip_seed22_n100000.csv
Detected separator: ';'
Loaded shape: (100003, 66)


,transaction_id,bank_code,bank_name,bank_country,bank_tier,client_id,client_segment,channel,card_brand,pan_masked,...,amount_usd,is_international,distance_from_home_km,hour_local,day_of_week,approved,response_description,client_baseline_amount,client_home_city,is_fraud
0,7dd812b1-bd03-4d05-afc6-c318dcc9b651,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00001325,PLATINUM,POS,MASTERCARD,531270******3773,...,500.12,True,8717.0,20,Tue,True,Approved,2012.51,TARIJA,False
1,c08b49a6-889a-491a-a1f8-974526f7886d,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00000079,PRIVATE,ECOM,VISA,421250******5552,...,1898.93,False,4.9,20,Tue,True,Approved,1096.46,LAPAZ,False
2,b04f88bd-2e33-42e5-a3cf-d52ef22dd7d9,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00002344,INFINITE,ECOM,NaN,531270******6104,...,349.85,False,4.4,20,Tue,True,Approved,1528.37,SANTACRUZ,False
3,3a836c25-7a8c-473b-8141-3e84ba3f212d,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00002587,PLATINUM,ATM,VISA,479500******0288,...,345.58,True,3966.0,20,Tue,True,Approved,2483.34,SUCRE,False
4,be9956da-924f-4c68-aed8-f0c5d949e577,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00000087,PRIVATE,POS,VISA,479500******0249,...,118.90,False,348.0,20,Tue,True,NaN,1334.55,SUCRE,False


Target validation passed
is_fraud
0    95084
1     4919


## EDA basico

Esta seccion revisa tipos de datos, valores faltantes, duplicados, cardinalidad, balance de la columna objetivo, estadisticas descriptivas y graficas basicas.

In [5]:
# Guarda la grafica actual
def save_current_plot(file_name):
    path = PLOTS_DIR / file_name
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    return path


# Resume tipos nulos y cardinalidad
def summarize_columns(data, target_column):
    summary = pd.DataFrame(
        {
            "dtype": data.dtypes.astype(str),
            "missing_count": data.isna().sum(),
            "missing_rate": data.isna().mean(),
            "unique_count": data.nunique(dropna=False),
        }
    )
    if target_column in data.columns:
        summary["unique_rate"] = summary["unique_count"] / max(len(data), 1)
    return summary.sort_values(["missing_rate", "unique_count"], ascending=[False, False])


# Elige una columna de monto
def choose_basic_amount_column(data):
    preferred_columns = [
        "amount_usd",
        "amount_local",
        "amount_tx_currency",
        "de4_amount_transaction",
        "de6_amount_cardholder_billing",
    ]
    for column in preferred_columns:
        if column in data.columns:
            return column
    amount_columns = [column for column in data.columns if "amount" in column or column.endswith("_amt")]
    return amount_columns[0] if amount_columns else None


# Grafica la distribucion del target
def plot_target_distribution(data, target_column):
    counts = data[target_column].value_counts().sort_index()
    plt.figure()
    plt.bar(counts.index.astype(str), counts.values)
    plt.title("Target distribution")
    plt.xlabel("is_fraud")
    plt.ylabel("Count")
    for index, value in enumerate(counts.values):
        plt.text(index, value, str(value), ha="center", va="bottom")
    save_current_plot("target_distribution.png")


# Grafica la distribucion de montos
def plot_amount_distribution(data, amount_column):
    if amount_column is None:
        warnings.warn("No amount column found for amount distribution plot")
        return
    amount_values = pd.to_numeric(data[amount_column], errors="coerce").dropna()
    if amount_values.empty:
        warnings.warn("Amount column has no numeric values")
        return
    clipped_values = amount_values.clip(upper=amount_values.quantile(0.99))
    plt.figure()
    plt.hist(clipped_values, bins=50)
    plt.title("Amount distribution")
    plt.xlabel(amount_column)
    plt.ylabel("Count")
    save_current_plot("amount_distribution.png")


# Grafica la tasa de fraude por mes
def plot_basic_fraud_rate_by_month(data, target_column):
    candidate_columns = [column for column in data.columns if "datetime" in column or "date" in column or column.startswith("de7")]
    if not candidate_columns:
        warnings.warn("No date like column found for fraud rate by month plot")
        return
    date_column = candidate_columns[0]
    values = data[date_column].astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True).str.zfill(10)
    month_values = pd.to_numeric(values.str.slice(0, 2), errors="coerce")
    valid_mask = month_values.between(1, 12)
    if valid_mask.sum() == 0:
        warnings.warn("Date like column could not be converted to month")
        return
    monthly_summary = (
        pd.DataFrame({"month": month_values[valid_mask], target_column: data.loc[valid_mask, target_column]})
        .groupby("month")[target_column]
        .agg(["mean", "count"])
        .reset_index()
    )
    plt.figure()
    plt.plot(monthly_summary["month"], monthly_summary["mean"], marker="o")
    plt.title("Fraud rate by month")
    plt.xlabel("Month")
    plt.ylabel("Fraud rate")
    save_current_plot("fraud_rate_by_month.png")
    display(monthly_summary)


# Grafica fraude por categorias principales
def plot_top_category_fraud_rates(data, target_column):
    candidate_columns = [
        "channel",
        "de22_pos_entry_mode",
        "de25_pos_condition_code",
        "de18_merchant_category_code",
        "currency_tx_alpha",
        "de60_pos_terminal_type",
    ]
    for column in candidate_columns:
        if column not in data.columns:
            continue
        category_summary = (
            data.groupby(column, dropna=False)[target_column]
            .agg(["mean", "count"])
            .sort_values("count", ascending=False)
            .head(10)
            .reset_index()
        )
        display(category_summary)
        plt.figure()
        plt.bar(category_summary[column].astype(str), category_summary["mean"])
        plt.title(f"Fraud rate by {column}")
        plt.xlabel(column)
        plt.ylabel("Fraud rate")
        plt.xticks(rotation=45, ha="right")
        save_current_plot(f"fraud_rate_by_{column}.png")


column_summary = summarize_columns(raw_data, TARGET_COLUMN)
duplicate_count = int(raw_data.duplicated().sum())
target_balance = raw_data[TARGET_COLUMN].value_counts(normalize=False).rename("count").to_frame()
target_balance["rate"] = raw_data[TARGET_COLUMN].value_counts(normalize=True)

print(f"Duplicate rows: {duplicate_count}")
column_name_table = pd.Series(raw_data.columns.tolist(), name="column_name").to_frame()
display(column_name_table)
display(column_summary.head(30))
display(target_balance)
display(raw_data.select_dtypes(include=[np.number]).describe().T)
display(raw_data.select_dtypes(exclude=[np.number]).describe().T.head(30))

column_summary.to_csv(EDA_DIR / "eda_column_summary.csv")
target_balance.to_csv(EDA_DIR / "eda_target_balance.csv")

basic_amount_column = choose_basic_amount_column(raw_data)
plot_target_distribution(raw_data, TARGET_COLUMN)
plot_amount_distribution(raw_data, basic_amount_column)
plot_basic_fraud_rate_by_month(raw_data, TARGET_COLUMN)
plot_top_category_fraud_rates(raw_data, TARGET_COLUMN)

Duplicate rows: 0


,column_name
0,transaction_id
1,bank_code
2,bank_name
3,bank_country
4,bank_tier
...,...
61,approved
62,response_description
63,client_baseline_amount
64,client_home_city


,dtype,missing_count,missing_rate,unique_count,unique_rate
de48_additional_data,float64,100003,1.000000,1,0.000010
de54_additional_amounts,float64,100003,1.000000,1,0.000010
de56_original_data,float64,100003,1.000000,1,0.000010
de103_account_id_2,float64,100003,1.000000,1,0.000010
de44_additional_response_data,object,96736,0.967331,11,0.000110
de38_authorization_code,object,3267,0.032669,96736,0.967331
de100_receiving_institution_id,float64,1050,0.010500,2,0.000020
client_segment,object,1046,0.010460,5,0.000050
de63_network_specific,object,1045,0.010450,2,0.000020
card_brand,object,1042,0.010420,3,0.000030


,count,rate
is_fraud,,
0,95084,0.950811
1,4919,0.049189


,count,mean,std,min,25%,50%,75%,max
mti,100003.0,1.000000e+02,0.000000e+00,1.000000e+02,1.000000e+02,1.000000e+02,1.000000e+02,1.000000e+02
de2_pan,100003.0,4.770996e+15,4.498077e+14,4.212500e+15,4.212510e+15,4.795000e+15,5.312700e+15,5.312710e+15
de3_processing_code,100003.0,3.424797e+03,2.339125e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.000000e+05
de4_amount_transaction,100003.0,1.426779e+06,1.154981e+07,7.000000e+00,3.704750e+04,1.112500e+05,3.013400e+05,4.911819e+08
de6_amount_cardholder_billing,100003.0,2.971216e+05,4.125483e+05,5.000000e+01,5.813150e+04,1.382990e+05,3.339170e+05,4.645012e+06
de7_transmission_datetime,100003.0,3.650214e+08,1.708106e+08,1.010002e+08,2.142012e+08,3.312233e+08,5.152135e+08,6.300013e+08
de9_conversion_rate_billing,99002.0,2.162929e+07,2.567151e+07,7.262900e+04,1.000000e+07,1.000000e+07,1.000000e+07,8.935550e+07
de11_stan,100003.0,8.371148e+05,3.155299e+05,0.000000e+00,9.249975e+05,9.499980e+05,9.749985e+05,9.999990e+05
de12_local_time,100003.0,1.328865e+05,5.880718e+04,2.000000e+00,9.130400e+04,1.332170e+05,1.805480e+05,2.359590e+05
de13_local_date,100003.0,3.654574e+02,1.726527e+02,1.010000e+02,2.140000e+02,3.310000e+02,5.150000e+02,1.231000e+03


...

,de22_pos_entry_mode,mean,count
0,81,0.080217,46312
1,51,0.017583,30427
2,71,0.049763,6129
3,10,0.010479,6012
4,72,0.010664,5814
5,21,0.010352,5120
6,22,0.984127,189


,de25_pos_condition_code,mean,count
0,0,0.023931,47679
1,59,0.080217,46312
2,8,0.010542,4838
3,1,0.010221,1174


,de18_merchant_category_code,mean,count
0,5732,0.060447,11018
1,6011,0.028380,9725
2,4511,0.045039,9192
3,7011,0.045786,9064
4,5311,0.009920,7863
5,5651,0.010004,7697
6,5967,0.105871,5979
7,4816,0.121622,5106
8,5944,0.075844,4918
9,5812,0.007512,4526


,currency_tx_alpha,mean,count
0,BOB,0.038348,76562
1,EUR,0.084928,9302
2,USD,0.083144,7024
3,JPY,0.089607,2444
4,GBP,0.079729,2358
5,CLP,0.087332,2313


,de60_pos_terminal_type,mean,count
0,ECOM-VIRTUAL,0.080217,46312
1,POS-ATTENDED,0.022414,39128
2,ATM-UNATTENDED,0.028380,9725
3,MOTO-CLERK,0.010542,4838


## Ayudantes de deteccion de columnas

Esta seccion detecta columnas relevantes para fecha, monto, cliente, tarjeta, comercio, MCC, pais, moneda, canal, modo de entrada POS, dispositivo y senales de comercio electronico o app.

In [ ]:
# Evalua coincidencias con palabras clave
def column_matches_keywords(column, keywords):
    column_text = column.lower()
    return any(keyword in column_text for keyword in keywords)


# Busca columnas por palabras clave
def find_keyword_matches(columns, keywords):
    return [column for column in columns if column_matches_keywords(column, keywords)]


# Elige la columna principal
def choose_primary_column(matches, preferred_columns):
    for preferred_column in preferred_columns:
        if preferred_column in matches:
            return preferred_column
    return matches[0] if matches else None


# Detecta columnas relevantes
def detect_relevant_columns(data):
    columns = list(data.columns)
    config = {
        "date_column": {
            "keywords": ["datetime", "timestamp", "date", "transmission", "local_date", "settlement", "de7", "de13"],
            "preferred": ["transaction_datetime", "de7_transmission_datetime", "de13_local_date", "de15_settlement_date"],
        },
        "amount_column": {
            "keywords": ["amount", "amt", "de4", "de6"],
            "preferred": ["amount_usd", "amount_local", "amount_tx_currency", "de4_amount_transaction"],
        },
        "customer_column": {
            "keywords": ["customer", "client", "cardholder", "account_id_1", "de102"],
            "preferred": ["client_id", "customer_id", "de102_account_id_1"],
        },
        "card_column": {
            "keywords": ["card", "pan", "account", "de2"],
            "preferred": ["pan_hash", "pan_masked", "de2_pan", "card_id"],
        },
        "merchant_column": {
            "keywords": ["merchant", "acceptor", "de42", "de43"],
            "preferred": ["de42_card_acceptor_id", "merchant_id", "de43_card_acceptor_name_location"],
        },
        "mcc_column": {
            "keywords": ["mcc", "merchant_category", "de18"],
            "preferred": ["de18_merchant_category_code", "mcc"],
        },
        "country_column": {
            "keywords": ["country", "acquirer_country", "merchant_country", "bank_country"],
            "preferred": ["de19_acquirer_country_code", "bank_country", "merchant_country", "cardholder_country"],
        },
        "currency_column": {
            "keywords": ["currency", "de49", "de50", "de51"],
            "preferred": ["currency_tx_alpha", "de49_currency_code_transaction", "de51_currency_code_billing"],
        },
        "channel_column": {
            "keywords": ["channel", "terminal_type", "de60"],
            "preferred": ["channel", "de60_pos_terminal_type"],
        },
        "pos_entry_mode_column": {
            "keywords": ["pos_entry", "entry_mode", "de22", "pos_data_code", "de123"],
            "preferred": ["de22_pos_entry_mode", "de123_pos_data_code"],
        },
        "device_column": {
            "keywords": ["device", "terminal", "wallet", "mobile", "pos_terminal", "de41", "de60"],
            "preferred": ["device_id", "de60_pos_terminal_type", "de41_terminal_id"],
        },
        "ecommerce_app_related_columns": {
            "keywords": ["is_mobile_app", "mobile_app", "app_origin", "app_channel", "mobile_origin", "device_type", "device_channel", "wallet_type", "digital_wallet", "mobile", "wallet", "ecom", "online", "virtual", "cnp", "pos_condition", "de25", "de60", "de123", "channel"],
            "preferred": ["channel", "de60_pos_terminal_type", "de25_pos_condition_code", "de123_pos_data_code"],
        },
    }
    detected = {}
    for group_name, group_config in config.items():
        matches = find_keyword_matches(columns, group_config["keywords"])
        primary = choose_primary_column(matches, group_config["preferred"])
        detected[group_name] = {"primary": primary, "candidates": matches}
    return detected


# Obtiene la columna principal detectada
def get_primary_column(detected_columns, group_name):
    group_value = detected_columns.get(group_name, {})
    return group_value.get("primary")

detected_columns = detect_relevant_columns(raw_data)
detected_rows = []
for group_name, group_value in detected_columns.items():
    detected_rows.append(
        {
            "group": group_name,
            "primary": group_value.get("primary"),
            "candidates": ", ".join(group_value.get("candidates", [])),
        }
    )

detected_columns_table = pd.DataFrame(detected_rows)
display(detected_columns_table)

with open(DATA_CHECKS_DIR / "detected_columns.json", "w", encoding="utf-8") as file:
    json.dump(detected_columns, file, indent=2, ensure_ascii=True)

,group,primary,candidates
0,date_column,de7_transmission_datetime,"de7_transmission_datetime, de13_local_date, de..."
1,amount_column,amount_usd,"de4_amount_transaction, de6_amount_cardholder_..."
2,customer_column,client_id,"client_id, client_segment, de6_amount_cardhold..."
3,card_column,pan_hash,"card_brand, pan_masked, pan_hash, de2_pan, de6..."
4,merchant_column,de42_card_acceptor_id,"de18_merchant_category_code, de42_card_accepto..."
5,mcc_column,de18_merchant_category_code,de18_merchant_category_code
6,country_column,de19_acquirer_country_code,"bank_country, de19_acquirer_country_code"
7,currency_column,currency_tx_alpha,"de49_currency_code_transaction, de50_currency_..."
8,channel_column,channel,"channel, de60_pos_terminal_type"
9,pos_entry_mode_column,de22_pos_entry_mode,"de22_pos_entry_mode, de123_pos_data_code"


## Identificacion de transacciones de app movil

Esta seccion infiere transacciones de app movil como una proxy conservadora basada en senales ecommerce y terminal virtual. La variable is_mobile_app_txn no es una etiqueta real de origen app movil porque el dataset no contiene una columna explicita confiable para ese origen.

In [7]:
# Normaliza valores de senales
def normalize_signal_values(data, column):
    if column is None or column not in data.columns:
        return pd.Series("", index=data.index)
    return data[column].astype(str).str.replace(r"\.0$", "", regex=True).str.upper().str.strip()


# Construye senales estrictas de app movil
def build_strict_app_signal(data):
    strict_app_columns = [
        "is_mobile_app",
        "mobile_app",
        "app_origin",
        "app_channel",
        "mobile_origin",
        "device_type",
        "device_channel",
        "wallet_type",
        "digital_wallet",
        "mobile_app_origin",
    ]
    available_app_columns = [column for column in strict_app_columns if column in data.columns]
    direct_app_signal = pd.Series(False, index=data.index)
    positive_pattern = r"MOBILE_APP|MOBILE APP|IN_APP|INAPP|APP|MOBILE|WALLET|DIGITAL_WALLET|IOS|ANDROID"

    for column in available_app_columns:
        values = normalize_signal_values(data, column)
        boolean_signal = values.isin(["1", "TRUE", "YES", "Y"])
        text_signal = values.str.contains(positive_pattern, regex=True, na=False)
        direct_app_signal = direct_app_signal | boolean_signal | text_signal

    return direct_app_signal, available_app_columns


# Crea la proxy de app movil
def infer_mobile_app_flag(input_data, detected_columns):
    result = input_data.copy()
    channel_column = "channel" if "channel" in result.columns else get_primary_column(detected_columns, "channel_column")
    terminal_column = "de60_pos_terminal_type" if "de60_pos_terminal_type" in result.columns else get_primary_column(detected_columns, "device_column")
    pos_entry_column = "de22_pos_entry_mode" if "de22_pos_entry_mode" in result.columns else get_primary_column(detected_columns, "pos_entry_mode_column")
    condition_column = "de25_pos_condition_code" if "de25_pos_condition_code" in result.columns else None

    # This flag is a conservative proxy, not a true mobile app origin label.
    direct_app_signal, available_app_columns = build_strict_app_signal(result)

    # Conservative ecommerce signals for the mobile app proxy.
    ecommerce_signals = []

    if channel_column in result.columns:
        is_ecom_channel = normalize_signal_values(result, channel_column).eq("ECOM")
        ecommerce_signals.append(is_ecom_channel.rename("is_ecom_channel"))
    else:
        is_ecom_channel = pd.Series(False, index=result.index)

    if terminal_column in result.columns:
        is_ecom_virtual_terminal = normalize_signal_values(result, terminal_column).eq("ECOM-VIRTUAL")
        ecommerce_signals.append(is_ecom_virtual_terminal.rename("is_ecom_virtual_terminal"))
    else:
        is_ecom_virtual_terminal = pd.Series(False, index=result.index)

    if pos_entry_column in result.columns:
        is_ecom_entry_mode = normalize_signal_values(result, pos_entry_column).isin(["81", "081"])
        ecommerce_signals.append(is_ecom_entry_mode.rename("is_ecom_entry_mode"))
    else:
        is_ecom_entry_mode = pd.Series(False, index=result.index)

    if condition_column in result.columns:
        is_ecom_condition_code = normalize_signal_values(result, condition_column).isin(["59", "059"])
        ecommerce_signals.append(is_ecom_condition_code.rename("is_ecom_condition_code"))
    else:
        is_ecom_condition_code = pd.Series(False, index=result.index)

    result["has_mobile_signal"] = direct_app_signal.astype(int)
    result["has_app_signal"] = direct_app_signal.astype(int)
    result["is_ecom_channel_signal"] = is_ecom_channel.astype(int)
    result["is_ecom_virtual_terminal_signal"] = is_ecom_virtual_terminal.astype(int)
    result["is_ecom_entry_mode_signal"] = is_ecom_entry_mode.astype(int)
    result["is_ecom_condition_code_signal"] = is_ecom_condition_code.astype(int)

    if ecommerce_signals:
        ecommerce_signal_frame = pd.concat(ecommerce_signals, axis=1)
        ecommerce_signal_count = ecommerce_signal_frame.sum(axis=1)
        available_signal_count = len(ecommerce_signals)
        if available_signal_count >= 2:
            ecommerce_proxy = ecommerce_signal_count >= 2
        else:
            ecommerce_proxy = ecommerce_signal_count >= 1
    else:
        warnings.warn("No conservative ecommerce signals found for mobile app proxy.")
        ecommerce_signal_count = pd.Series(0, index=result.index)
        ecommerce_proxy = pd.Series(False, index=result.index)

    # Explicit app fields are only used when their names are strict and reliable.
    if available_app_columns:
        mobile_app_flag = direct_app_signal | ecommerce_proxy
    else:
        mobile_app_flag = ecommerce_proxy

    result["mobile_app_signal_count"] = ecommerce_signal_count.astype(int)
    result["has_digital_channel_signal"] = ecommerce_proxy.astype(int)
    result[MOBILE_APP_COLUMN] = mobile_app_flag.astype(int)
    return result


# Aplica la proxy de app movil
working_data = infer_mobile_app_flag(raw_data, detected_columns)

mobile_counts = working_data[MOBILE_APP_COLUMN].value_counts().rename("count").to_frame()
mobile_counts["rate"] = working_data[MOBILE_APP_COLUMN].value_counts(normalize=True)
mobile_fraud_rate = (
    working_data.groupby(MOBILE_APP_COLUMN)[TARGET_COLUMN]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "fraud_rate", "sum": "fraud_count"})
)

display(mobile_counts)
display(mobile_fraud_rate)

mobile_fraud_rate.to_csv(EDA_DIR / "mobile_app_segment_summary.csv", index=True)

,count,rate
is_mobile_app_txn,,
0,53691,0.536894
1,46312,0.463106


,fraud_rate,fraud_count,count
is_mobile_app_txn,,,
0,0.022425,1204,53691
1,0.080217,3715,46312


## Division temporal

Esta seccion crea variables de fecha y hora, usa datos antes de junio 2025 para entrenamiento y usa junio 2025 como conjunto de prueba principal.

In [8]:
# Limpia texto numerico
def clean_numeric_text(series):
    return series.astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True)


# Convierte fecha iso8583 de7
def parse_iso8583_de7(series, assumed_year):
    values = clean_numeric_text(series).str.zfill(10)
    parts = pd.DataFrame(index=series.index)
    parts["year"] = assumed_year
    parts["month"] = pd.to_numeric(values.str.slice(0, 2), errors="coerce")
    parts["day"] = pd.to_numeric(values.str.slice(2, 4), errors="coerce")
    parts["hour"] = pd.to_numeric(values.str.slice(4, 6), errors="coerce")
    parts["minute"] = pd.to_numeric(values.str.slice(6, 8), errors="coerce")
    parts["second"] = pd.to_numeric(values.str.slice(8, 10), errors="coerce")
    return pd.to_datetime(parts, errors="coerce")


# Convierte fecha iso8583 de13
def parse_iso8583_de13(series, assumed_year):
    values = clean_numeric_text(series).str.zfill(4)
    parts = pd.DataFrame(index=series.index)
    parts["year"] = assumed_year
    parts["month"] = pd.to_numeric(values.str.slice(0, 2), errors="coerce")
    parts["day"] = pd.to_numeric(values.str.slice(2, 4), errors="coerce")
    parts["hour"] = 0
    parts["minute"] = 0
    parts["second"] = 0
    return pd.to_datetime(parts, errors="coerce")


# Convierte columnas candidatas a fecha
def parse_datetime_candidate(data, column, assumed_year):
    if column is None or column not in data.columns:
        return pd.Series(pd.NaT, index=data.index)
    lower_column = column.lower()
    if lower_column.startswith("de7") or "transmission_datetime" in lower_column:
        return parse_iso8583_de7(data[column], assumed_year)
    if lower_column.startswith("de13") or "local_date" in lower_column:
        return parse_iso8583_de13(data[column], assumed_year)
    parsed = pd.to_datetime(data[column], errors="coerce")
    valid_rate = parsed.notna().mean()
    if valid_rate >= 0.50:
        return parsed
    if "date" in lower_column:
        return parse_iso8583_de13(data[column], assumed_year)
    return parsed


# Construye la fecha de transaccion
def build_transaction_datetime(data, detected_columns, assumed_year):
    date_candidates = detected_columns.get("date_column", {}).get("candidates", [])
    if not date_candidates:
        warnings.warn("No date column detected.")
        return data.copy(), None, {}

    best_column = None
    best_parsed = None
    best_valid_rate = -1.0
    parse_report = {}

    for column in date_candidates:
        parsed = parse_datetime_candidate(data, column, assumed_year)
        valid_rate = float(parsed.notna().mean())
        parse_report[column] = valid_rate
        if valid_rate > best_valid_rate:
            best_column = column
            best_parsed = parsed
            best_valid_rate = valid_rate

    result = data.copy()
    if best_column is None or best_valid_rate <= 0:
        warnings.warn("Date parsing failed for detected date columns.")
        return result, None, parse_report

    result["transaction_datetime"] = best_parsed
    print(f"Selected date column: {best_column}")
    print(f"Date valid rate: {best_valid_rate:.4f}")
    return result, best_column, parse_report


# Agrega variables de calendario
def add_calendar_features(data):
    result = data.copy()
    if "transaction_datetime" in result.columns and result["transaction_datetime"].notna().any():
        result["year"] = result["transaction_datetime"].dt.year
        result["month"] = result["transaction_datetime"].dt.month
        result["day"] = result["transaction_datetime"].dt.day
        result["dayofweek"] = result["transaction_datetime"].dt.dayofweek
        result["hour"] = result["transaction_datetime"].dt.hour
    else:
        warnings.warn("Calendar features not created because transaction_datetime is unavailable.")
    return result


# Crea el split temporal
def create_time_split(data, datetime_column):
    result = data.copy()
    if datetime_column is None or "transaction_datetime" not in result.columns:
        warnings.warn("No datetime column available. Using row order split as fallback.")
        ordered_index = result.index.to_numpy()
        cut_position = int(len(ordered_index) * 0.80)
        return ordered_index[:cut_position], ordered_index[cut_position:], "row_order_fallback"

    june_start = pd.Timestamp("2025-06-01")
    july_start = pd.Timestamp("2025-07-01")
    january_start = pd.Timestamp("2025-01-01")

    date_values = result["transaction_datetime"]
    train_mask = (date_values >= january_start) & (date_values < june_start)
    test_mask = (date_values >= june_start) & (date_values < july_start)

    if train_mask.sum() == 0:
        warnings.warn("No January to May rows found. Using all rows before June as train.")
        train_mask = date_values < june_start

    if test_mask.sum() == 0:
        warnings.warn("No June 2025 rows found. Using last 20 percent by time as fallback.")
        ordered_index = result.sort_values("transaction_datetime").index.to_numpy()
        cut_position = int(len(ordered_index) * 0.80)
        return ordered_index[:cut_position], ordered_index[cut_position:], "time_order_fallback"

    train_index = result.index[train_mask].to_numpy()
    test_index = result.index[test_mask].to_numpy()

    june_train_mask = result.loc[train_index, "transaction_datetime"].between(june_start, july_start, inclusive="left")
    if bool(june_train_mask.any()):
        raise ValueError("Train split contains June 2025 rows.")

    if len(train_index) == 0 or len(test_index) == 0:
        raise ValueError("Train or test split is empty.")

    return train_index, test_index, "january_to_may_train_june_test"


working_data, selected_date_column, date_parse_report = build_transaction_datetime(working_data, detected_columns, ASSUMED_YEAR)
working_data = add_calendar_features(working_data)

train_index, test_index, split_strategy = create_time_split(working_data, selected_date_column)
train_data = working_data.loc[train_index].copy()
test_data = working_data.loc[test_index].copy()

split_summary = {
    "dataset_file": dataset_path.name,
    "selected_date_column": selected_date_column,
    "split_strategy": split_strategy,
    "train_rows": int(len(train_data)),
    "test_rows": int(len(test_data)),
    "train_start": str(train_data["transaction_datetime"].min()) if "transaction_datetime" in train_data.columns else None,
    "train_end": str(train_data["transaction_datetime"].max()) if "transaction_datetime" in train_data.columns else None,
    "test_start": str(test_data["transaction_datetime"].min()) if "transaction_datetime" in test_data.columns else None,
    "test_end": str(test_data["transaction_datetime"].max()) if "transaction_datetime" in test_data.columns else None,
    "date_parse_report": date_parse_report,
}

with open(DATA_CHECKS_DIR / "split_summary.json", "w", encoding="utf-8") as file:
    json.dump(split_summary, file, indent=2, ensure_ascii=True)

print(json.dumps(split_summary, indent=2))
display(train_data[TARGET_COLUMN].value_counts().rename("train_count").to_frame())
display(test_data[TARGET_COLUMN].value_counts().rename("test_count").to_frame())

Selected date column: de7_transmission_datetime
Date valid rate: 1.0000
{
  "dataset_file": "Copia de 01_bo_vip_seed22_n100000.csv",
  "selected_date_column": "de7_transmission_datetime",
  "split_strategy": "january_to_may_train_june_test",
  "train_rows": 83817,
  "test_rows": 16186,
  "train_start": "2025-01-01 00:01:51",
  "train_end": "2025-05-31 23:58:18",
  "test_start": "2025-06-01 00:00:04",
  "test_end": "2025-06-30 00:13:36",
  "date_parse_report": {
    "de7_transmission_datetime": 1.0,
    "de13_local_date": 1.0,
    "de14_expiration_date": 1.0,
    "de15_settlement_date": 0.9901802945911623,
    "de50_currency_code_settlement": 1.0
  }
}


,train_count
is_fraud,
0,79634
1,4183


,test_count
is_fraud,
0,15450
1,736


## Ingenieria de variables

Esta seccion crea variables de tiempo, cliente, tarjeta, comercio, monto, pais, fin de semana, noche y app movil sin fuga de la columna objetivo.

In [9]:
# Convierte valores a numerico seguro
def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")


# Agrega variables booleanas basicas
def add_basic_boolean_features(data, detected_columns):
    result = data.copy()
    amount_column = get_primary_column(detected_columns, "amount_column")

    if "is_international" in result.columns:
        result["is_foreign_txn"] = result["is_international"].astype(str).str.lower().isin(["true", "1", "yes"]).astype(int)
    else:
        country_column = get_primary_column(detected_columns, "country_column")
        if country_column and "bank_country" in result.columns:
            result["is_foreign_txn"] = (result[country_column].astype(str) != result["bank_country"].astype(str)).astype(int)
        else:
            result["is_foreign_txn"] = 0

    if "hour" in result.columns:
        hour_values = safe_numeric(result["hour"])
    elif "hour_local" in result.columns:
        hour_values = safe_numeric(result["hour_local"])
    else:
        hour_values = pd.Series(np.nan, index=result.index)

    result["is_night_txn"] = ((hour_values <= 5) | (hour_values >= 22)).fillna(False).astype(int)

    if "dayofweek" in result.columns:
        day_values = safe_numeric(result["dayofweek"])
        result["is_weekend_txn"] = day_values.isin([5, 6]).astype(int)
    elif "day_of_week" in result.columns:
        day_text = result["day_of_week"].astype(str).str.lower()
        result["is_weekend_txn"] = day_text.isin(["sat", "sun", "saturday", "sunday"]).astype(int)
    else:
        result["is_weekend_txn"] = 0

    if amount_column and amount_column in result.columns:
        result[amount_column] = safe_numeric(result[amount_column])

    return result


# Calcula conteos previos por tiempo
def compute_prior_counts_from_times(time_values, window_minutes):
    if len(time_values) == 0:
        return np.array([], dtype=int)
    time_ns = time_values.astype("int64")
    window_ns = int(window_minutes * 60 * 1_000_000_000)
    left_positions = np.searchsorted(time_ns, time_ns - window_ns, side="left")
    current_positions = np.arange(len(time_ns))
    return current_positions - left_positions


# Agrega variables temporales por entidad
def add_entity_time_features(data, entity_column, datetime_column):
    result = data.copy()
    result["time_since_last_txn_min"] = np.nan
    result["txn_count_last_1h"] = 0
    result["txn_count_last_24h"] = 0

    if entity_column is None or entity_column not in result.columns:
        warnings.warn("Entity column not found for rolling transaction features.")
        return result
    if datetime_column not in result.columns or result[datetime_column].notna().sum() == 0:
        warnings.warn("Datetime column not found for rolling transaction features.")
        return result

    valid_data = result[[entity_column, datetime_column]].dropna().sort_values([entity_column, datetime_column])
    for _, group in valid_data.groupby(entity_column, sort=False):
        group_index = group.index
        group_times = group[datetime_column].to_numpy(dtype="datetime64[ns]")
        if len(group_index) == 0:
            continue
        time_diffs = pd.Series(group_times).diff().dt.total_seconds().to_numpy() / 60.0
        result.loc[group_index, "time_since_last_txn_min"] = time_diffs
        result.loc[group_index, "txn_count_last_1h"] = compute_prior_counts_from_times(group_times, 60)
        result.loc[group_index, "txn_count_last_24h"] = compute_prior_counts_from_times(group_times, 1440)

    return result


# Mapea conteos por grupo
def map_group_size(train_frame, target_frame, key_column, output_column):
    if key_column is None or key_column not in train_frame.columns:
        target_frame[output_column] = 0
        return target_frame
    counts = train_frame.groupby(key_column).size()
    target_frame[output_column] = target_frame[key_column].map(counts).fillna(0).astype(float)
    return target_frame


# Agrega agregados basados en train
def add_train_based_aggregate_features(full_data, train_index, detected_columns):
    result = full_data.copy()
    train_frame = result.loc[train_index].copy()

    amount_column = get_primary_column(detected_columns, "amount_column")
    customer_column = get_primary_column(detected_columns, "customer_column")
    card_column = get_primary_column(detected_columns, "card_column")
    merchant_column = get_primary_column(detected_columns, "merchant_column")
    country_column = get_primary_column(detected_columns, "country_column")

    if amount_column and amount_column in result.columns:
        result[amount_column] = safe_numeric(result[amount_column])
        train_frame[amount_column] = safe_numeric(train_frame[amount_column])
        global_amount_mean = float(train_frame[amount_column].mean())
        global_amount_median = float(train_frame[amount_column].median())
        global_amount_std = float(train_frame[amount_column].std()) if train_frame[amount_column].std() else 1.0
    else:
        global_amount_mean = 0.0
        global_amount_median = 1.0
        global_amount_std = 1.0

    result = map_group_size(train_frame, result, customer_column, "customer_historical_txn_count")
    result = map_group_size(train_frame, result, card_column, "card_historical_txn_count")
    result = map_group_size(train_frame, result, merchant_column, "merchant_historical_txn_count")

    if customer_column and customer_column in result.columns and amount_column and amount_column in result.columns:
        customer_amount_stats = train_frame.groupby(customer_column)[amount_column].agg(["mean", "median", "std"])
        result["customer_amount_mean"] = result[customer_column].map(customer_amount_stats["mean"]).fillna(global_amount_mean)
        result["customer_amount_median"] = result[customer_column].map(customer_amount_stats["median"]).fillna(global_amount_median)
        result["customer_amount_std"] = result[customer_column].map(customer_amount_stats["std"]).replace(0, np.nan).fillna(global_amount_std)
        result["amount_zscore_customer"] = (result[amount_column] - result["customer_amount_mean"]) / result["customer_amount_std"]
        result["amount_ratio_customer_mean"] = result[amount_column] / result["customer_amount_mean"].replace(0, np.nan)
        result["amount_ratio_customer_median"] = result[amount_column] / result["customer_amount_median"].replace(0, np.nan)
    else:
        result["customer_amount_mean"] = global_amount_mean
        result["customer_amount_median"] = global_amount_median
        result["customer_amount_std"] = global_amount_std
        result["amount_zscore_customer"] = 0.0
        result["amount_ratio_customer_mean"] = 1.0
        result["amount_ratio_customer_median"] = 1.0

    if customer_column and merchant_column and customer_column in result.columns and merchant_column in result.columns:
        unique_merchants = train_frame.groupby(customer_column)[merchant_column].nunique()
        result["customer_unique_merchants_count"] = result[customer_column].map(unique_merchants).fillna(0).astype(float)
    else:
        result["customer_unique_merchants_count"] = 0.0

    if customer_column and country_column and customer_column in result.columns and country_column in result.columns:
        unique_countries = train_frame.groupby(customer_column)[country_column].nunique()
        result["customer_unique_countries_count"] = result[customer_column].map(unique_countries).fillna(0).astype(float)
    else:
        result["customer_unique_countries_count"] = 0.0

    if amount_column and amount_column in result.columns:
        mobile_amount_threshold = train_frame.loc[train_frame[MOBILE_APP_COLUMN] == 1, amount_column].quantile(0.90)
        if pd.isna(mobile_amount_threshold):
            mobile_amount_threshold = train_frame[amount_column].quantile(0.90)
        result["mobile_app_amount"] = result[amount_column].fillna(0) * result[MOBILE_APP_COLUMN].fillna(0)
        result["mobile_app_high_amount_flag"] = ((result[MOBILE_APP_COLUMN] == 1) & (result[amount_column] > mobile_amount_threshold)).astype(int)
    else:
        result["mobile_app_amount"] = 0.0
        result["mobile_app_high_amount_flag"] = 0

    if customer_column and customer_column in result.columns:
        customer_mobile_count = train_frame.groupby(customer_column)[MOBILE_APP_COLUMN].sum()
        result["customer_mobile_app_txn_count"] = result[customer_column].map(customer_mobile_count).fillna(0).astype(float)

        if amount_column and amount_column in train_frame.columns:
            customer_mobile_amount_mean = (
                train_frame.loc[train_frame[MOBILE_APP_COLUMN] == 1]
                .groupby(customer_column)[amount_column]
                .mean()
            )
            result["customer_mobile_app_amount_mean"] = result[customer_column].map(customer_mobile_amount_mean).fillna(global_amount_mean)
        else:
            result["customer_mobile_app_amount_mean"] = 0.0

        proxy_frame = train_frame.copy()
        proxy_frame["proxy_mobile_share"] = proxy_frame[MOBILE_APP_COLUMN].fillna(0)
        proxy_frame["proxy_foreign_share"] = proxy_frame["is_foreign_txn"].fillna(0) if "is_foreign_txn" in proxy_frame.columns else 0
        proxy_frame["proxy_high_amount_share"] = proxy_frame["mobile_app_high_amount_flag"].fillna(0) if "mobile_app_high_amount_flag" in proxy_frame.columns else 0
        customer_proxy = proxy_frame.groupby(customer_column)[["proxy_mobile_share", "proxy_foreign_share", "proxy_high_amount_share"]].mean()
        proxy_score = (
            0.50 * customer_proxy["proxy_mobile_share"]
            + 0.30 * customer_proxy["proxy_foreign_share"]
            + 0.20 * customer_proxy["proxy_high_amount_share"]
        )
        result["customer_mobile_app_fraud_risk_proxy"] = result[customer_column].map(proxy_score).fillna(0).astype(float)
    else:
        result["customer_mobile_app_txn_count"] = 0.0
        result["customer_mobile_app_amount_mean"] = 0.0
        result["customer_mobile_app_fraud_risk_proxy"] = 0.0

    ratio_columns = ["amount_ratio_customer_mean", "amount_ratio_customer_median"]
    for column in ratio_columns:
        result[column] = result[column].replace([np.inf, -np.inf], np.nan).fillna(1.0)
    result["amount_zscore_customer"] = result["amount_zscore_customer"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return result


entity_column = get_primary_column(detected_columns, "customer_column") or get_primary_column(detected_columns, "card_column")
feature_data = add_basic_boolean_features(working_data, detected_columns)
feature_data = add_entity_time_features(feature_data, entity_column, "transaction_datetime")
feature_data = add_train_based_aggregate_features(feature_data, train_index, detected_columns)

train_model_data = feature_data.loc[train_index].copy()
test_model_data = feature_data.loc[test_index].copy()

engineered_feature_columns = [
    "time_since_last_txn_min",
    "txn_count_last_1h",
    "txn_count_last_24h",
    "amount_zscore_customer",
    "amount_ratio_customer_mean",
    "amount_ratio_customer_median",
    "customer_historical_txn_count",
    "card_historical_txn_count",
    "merchant_historical_txn_count",
    "customer_unique_merchants_count",
    "customer_unique_countries_count",
    "is_foreign_txn",
    "is_night_txn",
    "is_weekend_txn",
    "mobile_app_amount",
    "mobile_app_high_amount_flag",
    "customer_mobile_app_txn_count",
    "customer_mobile_app_amount_mean",
    "customer_mobile_app_fraud_risk_proxy",
]

display(feature_data[engineered_feature_columns].head())

,time_since_last_txn_min,txn_count_last_1h,txn_count_last_24h,amount_zscore_customer,amount_ratio_customer_mean,amount_ratio_customer_median,customer_historical_txn_count,card_historical_txn_count,merchant_historical_txn_count,customer_unique_merchants_count,customer_unique_countries_count,is_foreign_txn,is_night_txn,is_weekend_txn,mobile_app_amount,mobile_app_high_amount_flag,customer_mobile_app_txn_count,customer_mobile_app_amount_mean,customer_mobile_app_fraud_risk_proxy
0,NaN,0,0,0.072964,1.101693,2.038145,20.0,20.0,163.0,17.0,6.0,1,1,0,0.00,0,6.0,698.406667,0.225000
1,NaN,0,0,2.751400,4.956012,14.598170,19.0,19.0,165.0,18.0,3.0,0,1,0,1898.93,1,10.0,602.622000,0.342105
2,NaN,0,0,-0.246887,0.697361,1.320687,13.0,13.0,163.0,13.0,2.0,0,1,0,349.85,0,6.0,641.118333,0.253846
3,NaN,0,0,-0.636174,0.318273,0.519185,23.0,23.0,175.0,23.0,7.0,1,1,0,0.00,0,9.0,1329.474444,0.313043
4,NaN,0,0,-0.717603,0.249702,0.429661,13.0,13.0,189.0,13.0,4.0,0,1,0,0.00,0,6.0,747.356667,0.300000


## Preprocesamiento

Esta seccion separa la columna objetivo y las variables predictoras, excluye campos no predictivos, maneja variables categoricas y crea la division de validacion.

In [10]:
# Crea el split de validacion
def make_train_validation_split(train_frame, target_column, datetime_column, valid_fraction=0.20):
    if datetime_column in train_frame.columns and train_frame[datetime_column].notna().sum() > 0:
        ordered_index = train_frame.sort_values(datetime_column).index.to_numpy()
        cut_position = int(len(ordered_index) * (1.0 - valid_fraction))
        train_sub_index = ordered_index[:cut_position]
        valid_index = ordered_index[cut_position:]
        if train_frame.loc[train_sub_index, target_column].nunique() == 2 and train_frame.loc[valid_index, target_column].nunique() == 2:
            return train_sub_index, valid_index, "time_validation"
        warnings.warn("Temporal validation split did not keep both classes. Using stratified fallback.")

    train_sub_parts = []
    valid_parts = []
    for _, group in train_frame.groupby(target_column):
        shuffled_index = group.sample(frac=1.0, random_state=RANDOM_STATE).index.to_numpy()
        valid_size = max(1, int(len(shuffled_index) * valid_fraction))
        valid_parts.append(shuffled_index[:valid_size])
        train_sub_parts.append(shuffled_index[valid_size:])

    train_sub_index = np.concatenate(train_sub_parts)
    valid_index = np.concatenate(valid_parts)
    return train_sub_index, valid_index, "stratified_random_validation"


# Decide si una variable se excluye
def should_exclude_feature(column, train_frame, target_column, raw_date_columns):
    if column == target_column:
        return True
    if column in raw_date_columns:
        return True
    if pd.api.types.is_datetime64_any_dtype(train_frame[column]):
        return True
    excluded_exact = {
        "transaction_id",
        "pan_masked",
        "pan_hash",
        "de2_pan",
        "de11_stan",
        "de35_track2_data_masked",
        "de37_retrieval_reference_number",
        "de38_authorization_code",
        "de102_account_id_1",
        "de103_account_id_2",
    }
    if column in excluded_exact:
        return True
    excluded_keywords = ["authorization_code", "retrieval_reference", "track2", "masked", "hash"]
    if any(keyword in column for keyword in excluded_keywords):
        return True
    unique_rate = train_frame[column].nunique(dropna=False) / max(len(train_frame), 1)
    if train_frame[column].dtype == "object" and unique_rate > 0.50:
        return True
    return False


# Prepara variables para LightGBM
def prepare_lightgbm_features(train_frame, test_frame, target_column, detected_columns, selected_date_column):
    raw_date_columns = {"transaction_datetime"}
    if selected_date_column:
        raw_date_columns.add(selected_date_column)

    candidate_features = []
    for column in train_frame.columns:
        if should_exclude_feature(column, train_frame, target_column, raw_date_columns):
            continue
        if train_frame[column].nunique(dropna=False) <= 1:
            continue
        candidate_features.append(column)

    x_train = train_frame[candidate_features].copy()
    x_test = test_frame[candidate_features].copy()
    categorical_features = []

    for column in candidate_features:
        if pd.api.types.is_bool_dtype(x_train[column]):
            x_train[column] = x_train[column].astype(int)
            x_test[column] = x_test[column].astype(int)
        elif pd.api.types.is_numeric_dtype(x_train[column]):
            x_train[column] = pd.to_numeric(x_train[column], errors="coerce")
            x_test[column] = pd.to_numeric(x_test[column], errors="coerce")
        else:
            train_text = x_train[column].astype("string").fillna("__missing__")
            test_text = x_test[column].astype("string").fillna("__missing__")
            categories = pd.Index(train_text.unique()).union(pd.Index(test_text.unique()))
            x_train[column] = pd.Categorical(train_text, categories=categories)
            x_test[column] = pd.Categorical(test_text, categories=categories)
            categorical_features.append(column)

    return x_train, x_test, categorical_features, candidate_features


train_sub_index, valid_index, validation_strategy = make_train_validation_split(
    train_model_data,
    TARGET_COLUMN,
    "transaction_datetime",
)

x_train_all, x_test, categorical_features, feature_columns = prepare_lightgbm_features(
    train_model_data,
    test_model_data,
    TARGET_COLUMN,
    detected_columns,
    selected_date_column,
)

y_train_all = train_model_data[TARGET_COLUMN].astype(int)
y_test = test_model_data[TARGET_COLUMN].astype(int)

x_train_sub = x_train_all.loc[train_sub_index].copy()
y_train_sub = y_train_all.loc[train_sub_index].copy()
x_valid = x_train_all.loc[valid_index].copy()
y_valid = y_train_all.loc[valid_index].copy()

with open(FEATURES_DIR / "feature_list.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            "feature_count": len(feature_columns),
            "features": feature_columns,
            "categorical_features": categorical_features,
            "validation_strategy": validation_strategy,
        },
        file,
        indent=2,
        ensure_ascii=True,
    )

print(f"Feature count: {len(feature_columns)}")
print(f"Categorical feature count: {len(categorical_features)}")
print(f"Validation strategy: {validation_strategy}")
display(pd.DataFrame({"feature": feature_columns, "is_categorical": [column in categorical_features for column in feature_columns]}).head(50))

Feature count: 76
Categorical feature count: 17
Validation strategy: time_validation


,feature,is_categorical
0,client_id,True
1,client_segment,True
2,channel,True
3,card_brand,True
4,de3_processing_code,False
5,de4_amount_transaction,False
6,de6_amount_cardholder_billing,False
7,de9_conversion_rate_billing,False
8,de12_local_time,False
9,de13_local_date,False


## Utilidades de evaluacion

Esta seccion define funciones reutilizables de evaluacion y graficas con false_positive_ratio como metrica central.

In [11]:
try:
    from sklearn.metrics import (
        average_precision_score,
        confusion_matrix,
        f1_score,
        precision_recall_curve,
        precision_score,
        recall_score,
        roc_auc_score,
        roc_curve,
    )
except ImportError as error:
    raise ImportError("scikit-learn is required. Run the commented install cell if needed.") from error


# Convierte scores a predicciones binarias
def get_binary_predictions(y_score, threshold):
    return (np.asarray(y_score) >= threshold).astype(int)


# Calcula conteos de confusion
def get_confusion_counts(y_true, y_pred):
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    return int(tn), int(fp), int(fn), int(tp)


# Calcula ratio de falsos positivos
def compute_false_positive_ratio(tp, fp):
    denominator = tp + fp
    if denominator == 0:
        return 0.0
    return fp / denominator


# Calcula auc roc seguro
def safe_auc_roc(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)


# Calcula auc pr seguro
def safe_auc_pr(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, y_score)


# Calcula metricas de clasificacion
def compute_classification_metrics(y_true, y_score, threshold):
    y_true_array = np.asarray(y_true).astype(int)
    y_score_array = np.asarray(y_score)
    y_pred = get_binary_predictions(y_score_array, threshold)
    tn, fp, fn, tp = get_confusion_counts(y_true_array, y_pred)

    precision_value = precision_score(y_true_array, y_pred, zero_division=0)
    recall_value = recall_score(y_true_array, y_pred, zero_division=0)
    f1_value = f1_score(y_true_array, y_pred, zero_division=0)
    specificity_value = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    false_positive_rate_value = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_positive_ratio_value = compute_false_positive_ratio(tp, fp)

    return {
        "auc_roc": safe_auc_roc(y_true_array, y_score_array),
        "auc_pr": safe_auc_pr(y_true_array, y_score_array),
        "f1": f1_value,
        "precision": precision_value,
        "recall": recall_value,
        "specificity": specificity_value,
        "false_positive_rate": false_positive_rate_value,
        "false_positive_ratio": false_positive_ratio_value,
        "true_positives": tp,
        "false_positives": fp,
        "true_negatives": tn,
        "false_negatives": fn,
        "threshold": float(threshold),
        "predicted_positives": int(tp + fp),
        "actual_positives": int(tp + fn),
    }


# Busca threshold para recall objetivo
def find_threshold_for_recall(y_true, y_score, target_recall=0.90):
    y_true_array = np.asarray(y_true).astype(int)
    y_score_array = np.asarray(y_score)
    positive_count = int(y_true_array.sum())
    if positive_count == 0:
        warnings.warn("No positive labels found. Returning threshold 1.0.")
        return 1.0

    order = np.argsort(-y_score_array)
    sorted_scores = y_score_array[order]
    sorted_true = y_true_array[order]

    unique_end_positions = np.r_[np.where(sorted_scores[:-1] != sorted_scores[1:])[0], len(sorted_scores) - 1]
    thresholds = sorted_scores[unique_end_positions]
    cumulative_true_positive = np.cumsum(sorted_true == 1)[unique_end_positions]
    predicted_positive = unique_end_positions + 1
    cumulative_false_positive = predicted_positive - cumulative_true_positive
    recall_values = cumulative_true_positive / positive_count
    false_positive_ratio_values = np.divide(
        cumulative_false_positive,
        predicted_positive,
        out=np.zeros_like(cumulative_false_positive, dtype=float),
        where=predicted_positive > 0,
    )

    candidate_mask = recall_values >= target_recall
    if candidate_mask.any():
        candidate_indices = np.where(candidate_mask)[0]
        best_local = candidate_indices[np.lexsort((-recall_values[candidate_indices], false_positive_ratio_values[candidate_indices]))][0]
        return float(thresholds[best_local])

    best_index = int(np.argmax(recall_values))
    warnings.warn("Target recall could not be reached. Using threshold with maximum recall.")
    return float(thresholds[best_index])


# Evalua al recall objetivo
def evaluate_at_target_recall(y_true, y_score, target_recall=0.90):
    threshold = find_threshold_for_recall(y_true, y_score, target_recall)
    metrics = compute_classification_metrics(y_true, y_score, threshold)
    metrics["target_recall"] = target_recall
    return metrics


# Evalua el segmento app movil
def evaluate_mobile_app_segment(y_true, y_score, mobile_flag, threshold):
    y_true_series = pd.Series(np.asarray(y_true).astype(int))
    y_score_series = pd.Series(np.asarray(y_score))
    mobile_series = pd.Series(np.asarray(mobile_flag).astype(int)).fillna(0).astype(int)

    rows = []
    segment_masks = {
        "all_test": np.ones(len(y_true_series), dtype=bool),
        "mobile_app": mobile_series.to_numpy() == 1,
        "non_mobile_app": mobile_series.to_numpy() == 0,
    }

    for segment_name, mask in segment_masks.items():
        if mask.sum() == 0:
            continue
        metrics = compute_classification_metrics(y_true_series[mask], y_score_series[mask], threshold)
        metrics["segment"] = segment_name
        metrics["rows"] = int(mask.sum())
        rows.append(metrics)
    return pd.DataFrame(rows)


# Evalua el modelo por segmentos
def evaluate_model_by_segments(model_name, metric_name, threshold_name, y_true, y_score, mobile_flag, threshold):
    segment_table = evaluate_mobile_app_segment(y_true, y_score, mobile_flag, threshold)
    segment_table.insert(0, "model_name", model_name)
    segment_table.insert(1, "custom_metric_key", metric_name)
    segment_table.insert(2, "threshold_name", threshold_name)
    return segment_table


# Grafica matriz de confusion
def plot_confusion_matrix(y_true, y_score, threshold, title, file_name):
    y_pred = get_binary_predictions(y_score, threshold)
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    plt.figure(figsize=(5, 4))
    plt.imshow(matrix, cmap="Blues")
    plt.title(title)
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.xticks([0, 1], ["0", "1"])
    plt.yticks([0, 1], ["0", "1"])
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            plt.text(col, row, str(matrix[row, col]), ha="center", va="center")
    save_current_plot(file_name)


# Grafica curva precision recall
def plot_precision_recall_curve(y_true, y_score, title, file_name):
    precision_values, recall_values, _ = precision_recall_curve(y_true, y_score)
    plt.figure()
    plt.plot(recall_values, precision_values)
    plt.title(title)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    save_current_plot(file_name)


# Grafica curva roc
def plot_roc_curve(y_true, y_score, title, file_name):
    if len(np.unique(y_true)) < 2:
        warnings.warn("ROC curve skipped because only one class is present.")
        return
    false_positive_rate_values, true_positive_rate_values, _ = roc_curve(y_true, y_score)
    plt.figure()
    plt.plot(false_positive_rate_values, true_positive_rate_values)
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.title(title)
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    save_current_plot(file_name)


# Grafica distribucion de scores
def plot_score_distribution(y_true, y_score, title, file_name):
    y_true_array = np.asarray(y_true).astype(int)
    y_score_array = np.asarray(y_score)
    plt.figure()
    plt.hist(y_score_array[y_true_array == 0], bins=50, alpha=0.6, label="Non fraud")
    plt.hist(y_score_array[y_true_array == 1], bins=50, alpha=0.6, label="Fraud")
    plt.title(title)
    plt.xlabel("Predicted score")
    plt.ylabel("Count")
    plt.legend()
    save_current_plot(file_name)